# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
 level=logging.INFO,
 format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
 datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
 SparkSession
 .builder
 .appName("SessaoLocal")
 .master("local[12]")
 .config("spark.driver.memory", "24g")
 .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_claro = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Claro/home")
_files_claro = [(str(file), str(file.parent.name)) for file in folder_claro.rglob("*PCL/*.gz") if file.is_file()]
df = pd.DataFrame(_files_claro, columns=["file_path", "parent_folder"])
df["exchange_id"] = df["parent_folder"].str.slice(stop=4)
files_claro = df.groupby("exchange_id").sample(15,random_state=42)["file_path"].to_list()
files_claro[:5]

### Arquivos intermediário e final

In [ ]:
extracted_claro = "/assets/data/cdr/teletools/db/extraidos/pci_claro_extraido.parquet"
transformed_claro = "/assets/data/cdr/teletools/db/transformados/pci_claro_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
df_raw = spark.read.csv(files_claro, sep=";", schema="value string")
dfx = df_raw.limit(5).toPandas()
dfx

In [ ]:
# planilha do Lannei
column_names = ("VERSAO", "TIPOARQUIVO", "TEC_FORNECEDOR", "ID_UNICO", "CENTRAL", "TIPOREGISTRO", "INDTAC", "RECORD_NUMBER", "SEQNUM (Sequencial do CDR)", "CC_A (Country Code do Número de A)", "CC_B (Country Code do Número de B)", "CC_C", "CC_MSRN", "CODAREAA", "CODAREAB", "CODAREAC", "CODAREAMSRN", "NPAIN", "NPAOUT", "CSP", "CSPC", "CSPMSRN", "TARIFCLASS", "DATE (Data da chamada no formato YYYYMMDD)", "TIME (Hora da chamada no formato HHMMSS)", "INTERRUPTIONTIME", "DUR (Duração da chamada em segundos)", "ANUM (Número de A original)", "BNUM (Número de B original)", "CNUM (Número de C original - Call Foward)", "MOBSTATIONRN", "ASUBNUM", "AUNKNOWN", "PREFIXNUMB", "BSUBNUM", "BUNKNOWN", "PREFIXNUMC", "CSUBNUM", "CUNKNOWN", "PREFIXMSRN", "MSRNSUBNUM", "MSRNUNKNOWN", "BSPECFLAG", "INROUTE", "TYPEROUTEIN", "OPERROTAENTRADA", "IN_DROPIND", "IN_SERVICEROUTE", "OUTROUTE", "TYPEROUTEOUT", "OPERROTASAIDA", "OUT_DROPIND", "OUT_SERVICEROUTE", "TYPECALL", "ABNORMALCALLRELEASE", "TYPESUBSA", "TYPESUBSB", "CALLSTATUS", "FLAG_RN", "EOT_A", "CNL_A", "EOT_B", "CNL_B", "EOT_C", "CNL_C", "EOT_MSRN", "CNL_MSRN", "EOT_TRANSLATED", "CNL_TRANSLATED", "EOT_ANT_A", "EOT_ANT_B", "EOT_ANT_C", "EOT_ANT_TRLTED", "EOT_IMSI", "IND_PORT", "DT_PORT_A", "DT_PORT_B", "DT_PORT_C", "DT_PORT_TRANSLATED", "CODOPERADORAA", "CODOPERADORAB", "CODOPERADORAHOM", "OPERATORTYPEA", "OPERATORTYPEB", "IMSIA (IMSI do Número de A)", "IMSIB (IMSI do Número de B)", "IMEIA (IMEI do Número de A)", "IMEIB (IMEI do Número de B)", "TYPEOFCALLINGSUBS", "TYPEOFCALLEDSUBS", "CALLIDNUMBER", "RELCALLNUM", "NETWORKCALLREF", "REDIRECTIMSI", "SMTTELESERVICECODE", "EOSINFO", "SMS_RESULT", "CALLPOSITION", "INFLAG", "INSERVICEKEY", "SSCODE", "TRANSLATEDNUM", "TUNKNOWN", "CC_TRANSLATED", "CODAREATRANSLATED", "CSP_TRANSLATED", "SUB_TRANSLATED", "PREFIX_TRANSLATED", "ORIGINATOR", "TERMINATOR", "CALL_TYPE", "COMPLETION_CODE", "SERV_TRANS_SCHEME", "CALLING_SYS_ID", "CALLED_SYS_ID", "COMPCD", "EVENTDIG", "TRMTCD", "CUST_TYPE", "SERVICE_CODE", "SUBSERVIND", "VM_RETRIEVAL_IND", "ACCEPT_CALL_WAIT", "INVOC_CALL_HOLD", "RET_HELD_CALL", "LAST_PARTIAL_OUT", "PARTIAL_OUT_REC_NUM", "CALL_FORWARD", "CALL_WAIT", "CALL_HOLD", "CALL_CONF", "ACATEGORY (Categoria A)", "BCATEGORY (Fim de Seleção)", "GMTOFFSET", "DESCITX", "COUNT226", "VIDEO_CALL", "SEQ_CDR", "FILLER")
column_indices = (1,2,5,8,24,32,34,44,46,56,59,62,65,68,70,72,74,76,78,80,82,84,86,90,98,104,110,116,138,166,188,214,232,233,235,253,254,256,274,275,277,295,296,297,305,306,309,310,313,321,322,325,326,329,333,334,335,336,337,338,341,346,349,354,357,362,365,370,373,378,381,384,387,390,393,394,402,410,418,426,428,430,432,433,434,450,466,481,496,498,499,509,519,529,545,547,549,550,551,553,555,557,585,586,589,591,593,611,613,627,641,642,644,647,652,657,659,660,663,666,667,669,671,672,673,674,675,676,677,678,679,680,682,684,686,689,691,692,703)
column_sizes = (1, 3, 3, 16, 8, 2, 10, 2, 10, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 8, 6, 6, 6, 22, 28, 22, 26, 18, 1, 2, 18, 1, 2, 18, 1, 2, 18, 1, 1, 8, 1, 3, 1, 3, 8, 1, 3, 1, 3, 4, 1, 1, 1, 1, 1, 3, 5, 3, 5, 3, 5, 3, 5, 3, 5, 3, 3, 3, 3, 3, 1, 8, 8, 8, 8, 2, 2, 2, 1, 1, 16, 16, 15, 15, 2, 1, 10, 10, 10, 16, 2, 2, 1, 1, 2, 2, 2, 28, 1, 3, 2, 2, 18, 2, 14, 14, 1, 2, 3, 5, 5, 2, 1, 3, 3, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 2, 1, 11, 11)

columns_expressions = [
    F.trim(
        F.regexp_replace(F.substring(F.col("value"), indice, size), "-", "")
    ).alias(name)
    for indice, size, name in zip(
        column_indices, column_sizes, column_names
    )
]
df_pcl = df_raw.select(*columns_expressions)
columns_plc = df_pcl.columns
df_pcl.limit(5).toPandas()

In [ ]:
# Planilha mais recente
column_names_2 = ("VERSAO", "TIPOARQUIVO", "TEC_FORNECEDOR", "ID_UNICO", "CENTRAL", "TIPOREGISTRO", "INDTAC", "RECORD_NUMBER", "SEQNUM", "CC_A", "CC_B", "CC_C", "CC_MSRN", "CODAREAA", "CODAREAB", "CODAREAC", "CODAREAMSRN", "NPAIN", "NPAOUT", "CSP", "CSPC", "CSPMSRN", "TARIFCLASS", "DATE", "TIME", "INTERRUPTIONTIME", "DUR", "ANUM", "BNUM", "CNUM", "MOBSTATIONRN", "ASUBNUM", "AUNKNOWN", "PREFIXNUMB", "BSUBNUM", "BUNKNOWN", "PREFIXNUMC", "CSUBNUM", "CUNKNOWN", "PREFIXMSRN", "MSRNSUBNUM", "MSRNUNKNOWN", "BSPECFLAG", "INROUTE", "TYPEROUTEIN", "OPERROTAENTRADA", "IN_DROPIND", "IN_SERVICEROUTE", "OUTROUTE", "TYPEROUTEOUT", "OPERROTASAIDA", "OUT_DROPIND", "OUT_SERVICEROUTE", "TYPECALL", "ABNORMALCALLRELEASE", "TYPESUBSA", "TYPESUBSB", "CALLSTATUS", "FLAG_RN", "EOT_A", "CNL_A", "EOT_B", "CNL_B", "EOT_C", "CNL_C", "EOT_MSRN", "CNL_MSRN", "EOT_TRANSLATED", "CNL_TRANSLATED", "EOT_ANT_A", "EOT_ANT_B", "EOT_ANT_C", "EOT_ANT_TRLTED", "EOT_IMSI", "IND_PORT", "DT_PORT_A", "DT_PORT_B", "DT_PORT_C", "DT_PORT_TRANSLATED", "CODOPERADORAA", "CODOPERADORAB", "CODOPERADORAHOM", "OPERATORTYPEA", "OPERATORTYPEB", "IMSIA", "IMSIB", "IMEIA", "IMEIB", "TYPEOFCALLINGSUBS", "TYPEOFCALLEDSUBS", "CALLIDNUMBER", "RELCALLNUM", "NETWORKCALLREF", "REDIRECTIMSI", "SMTTELESERVICECODE", "EOSINFO", "SMS_RESULT", "CALLPOSITION", "INFLAG", "INSERVICEKEY", "SSCODE", "TRANSLATEDNUM", "TUNKNOWN", "CC_TRANSLATED", "CODAREATRANSLATED", "CSP_TRANSLATED", "SUB_TRANSLATED", "PREFIX_TRANSLATED", "ORIGINATOR", "TERMINATOR", "CALL_TYPE", "COMPLETION_CODE", "SERV_TRANS_SCHEME", "CALLING_SYS_ID", "CALLED_SYS_ID", "COMPCD", "EVENTDIG", "TRMTCD", "CUST_TYPE = PT", "SERVICE_CODE", "SUBSERVIND", "VM_RETRIEVAL_IND", "ACCEPT_CALL_WAIT", "INVOC_CALL_HOLD", "RET_HELD_CALL", "LAST_PARTIAL_OUT", "PARTIAL_OUT_REC_NUM", "CALL_FORWARD", "CALL_WAIT", "CALL_HOLD", "CALL_CONF", "ACATEGORY", "BCATEGORY", "GMTOFFSET", "DESCITX", "COUNT226", "VIDEO_CALL", "SEQ_CDR", "FILLER")
column_indices_2 = (1,2,5,8,24,32,34,44,46,56,59,62,65,68,70,72,74,76,78,80,82,84,86,90,98,104,110,116,138,166,188,214,232,233,235,253,254,256,274,275,277,295,296,297,305,306,309,310,313,321,322,325,326,329,333,334,335,336,337,338,341,346,349,354,357,362,365,370,373,378,381,384,387,390,393,394,402,410,418,426,428,430,432,433,434,450,466,481,496,498,499,509,519,529,545,547,549,550,551,553,555,557,577,578,581,583,585,603,605,619,633,634,636,639,644,649,651,652,655,658,659,661,663,664,665,666,667,668,669,670,671,672,674,676,678,681,683,684,694)
column_sizes_2 = (1, 3, 3, 16, 8, 2, 10, 2, 10, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 8, 6, 6, 6, 22, 28, 22, 26, 18, 1, 2, 18, 1, 2, 18, 1, 2, 18, 1, 1, 8, 1, 3, 1, 3, 8, 1, 3, 1, 3, 4, 1, 1, 1, 1, 1, 3, 5, 3, 5, 3, 5, 3, 5, 3, 5, 3, 3, 3, 3, 3, 1, 8, 8, 8, 8, 2, 2, 2, 1, 1, 16, 16, 15, 15, 2, 1, 10, 10, 10, 16, 2, 2, 1, 1, 2, 2, 2, 20, 1, 3, 2, 2, 18, 2, 14, 14, 1, 2, 3, 5, 5, 2, 1, 3, 3, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 2, 1, 10, 12)

columns_expressions = [
    F.trim(
        F.regexp_replace(F.substring(F.col("value"), indice, size), "-", "")
    ).alias(name)
    for indice, size, name in zip(
        column_indices_2, column_sizes_2, column_names_2
    )
]
df_pcl2 = df_raw.select(*columns_expressions)
columns_plc2 = df_pcl2.columns
df_pcl2.limit(5).toPandas()

In [ ]:
indices_to_keep = (4,5,23,24,26,27,28,43,48,90,92,97)
columns_to_keep = [columns_plc[i] for i in indices_to_keep]
df_pcl.select(columns_to_keep).limit(5).toPandas()

In [ ]:
df_pcl.groupBy("TEC_FORNECEDOR").count().show()

In [ ]:
df_pcl.groupBy("TIPOREGISTRO").pivot("TEC_FORNECEDOR").count().orderBy("TIPOREGISTRO").show()

In [ ]:
df_pcl.filter(F.col("NETWORKCALLREF")=="").groupBy("TEC_FORNECEDOR").count().show()

In [ ]:
df_pcl.select(columns_to_keep).filter(F.col("NETWORKCALLREF")=="54DCD40006").dropDuplicates().show()

In [ ]:
pci_layout = [(column_indices, column_sizes, column_names) for column_indices, column_sizes, column_names in zip(column_indices, column_sizes, column_names)]
pci_layout_selected = [field for field in pci_layout if field[2] in columns_to_keep]
pci_layout_selected

In [ ]:
[ci for (ci, cs, cn) in pci_layout_selected]

In [ ]:
[cs for (ci, cs, cn) in pci_layout_selected]

In [ ]:
[cn for (ci, cs, cn) in pci_layout_selected]

In [ ]:
# df.groupBy(F.col("Nome da Central")).count().show()

In [ ]:
# df.groupBy(F.col("Tipo de Registro")).count().show()

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_claro, extracted_claro, "stfc_pcl_claro")

In [ ]:
df_extracted = spark.read.parquet(extracted_claro)
df_extracted.show(5, truncate=False)

In [ ]:
# 01 = MOC fabricante Nokia
# 02 = RCF fabricante Ericsson
# 03 = CF   fabricante Nokia ou Ericsson
# 04 = MTC fabricante Nokia"
df_extracted.groupBy("_tipo_chamada").pivot("tecnologia_fornecedor").count().orderBy("_tipo_chamada").show()

In [ ]:
# "3", "b-AnswerHasBeenReceived"
df_extracted.groupBy("_resultado_chamada").count().orderBy("_resultado_chamada").show()

In [ ]:
df_extracted.printSchema()

In [ ]:
df_extracted.groupBy("duracao").count().orderBy("duracao").show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_pcl_claro(extracted_claro, transformed_claro)

In [ ]:
df_transformed = spark.read.parquet(transformed_claro)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)

In [ ]:
df_transformed.groupBy("no_tipo_chamada").count().show()

In [ ]:
df_transformed.printSchema()